# Problem 1 (2 pts)

In real-world applications (such as neural network weight initialization or financial risk bounds), we often want variables that cluster around zero like a normal distribution, but strictly avoid extreme outliers. A **truncated normal distribution** $\mathcal{TN}(0, 1; -s, s)$ has a density proportional to $\exp(-x^2 / 2)$ on the interval $[-s, s]$ and zero elsewhere.

1. Implement a sampler using **naive rejection from a standard normal distribution** $\mathcal{N}(0, 1)$ (drawing from $\mathcal{N}(0, 1)$ and discarding values outside $[-s, s]$).
2. Implement a sampler using **rejection sampling with a uniform proposal** over $[-s, s]$. Determine the optimal scaling constant $M$ analytically.
3. Compare the empirical acceptance rates of both methods for $s = 0.1$ and $s = 3.0$. For which regimes of $s$ does each method dominate, and why?

In [3]:
# You're on your own, sorry!

# Problem 2 (2 pts)

## Markov Chain Monte Carlo (MCMC)

### Why move beyond rejection sampling?

Rejection sampling is conceptually elegant and exact: it is a *Las Vegas* algorithm that outputs independent samples with zero approximation error. However, it relies on finding a proposal density $g(x)$ and an envelope constant $M$ such that:

$$M g(x) \ge f(x) \quad \text{for all } x.$$

In practical scientific applications, this condition fails for two major reasons:

1. **The Curse of Dimensionality:** In $d$ dimensions, probability mass tends to concentrate in narrow, complex regions. The ratio of the target's volume to any simple bounding proposal's volume vanishes exponentially:

$$\mathbb{P}(\text{accept}) = \frac{1}{M} \sim \mathcal{O}(e^{-c \cdot d}).$$



For $d \ge 20$, the expected number of draws before a single acceptance can easily exceed the age of the universe.

2. **Intractable Geometry:** In Bayesian inference, posterior distributions $p(\theta \mid \mathcal{D}) \propto p(\mathcal{D} \mid \theta)p(\theta)$ often feature curved ridges and tight degeneracies. Constructing an analytical proposal $g(\theta)$ that tightly bounds such surfaces without knowing where the mass concentrates is practically impossible.

We need a method that does not attempt to enclose the entire distribution globally, but instead **navigates the high-probability landscape locally**.

---

### The core idea: Sampling along a trajectory

Instead of drawing independent points across the entire space and discarding the misses, MCMC constructs a **Markov chain** $\{X_0, X_1, X_2, \dots\}$ designed to have our target distribution $p(x) \propto f(x)$ as its unique **stationary distribution** (remember PageRank?).

* The sampler starts at an arbitrary point $X_0$.
* At each step $t$, it proposes a local perturbation $X^* \sim q(X^* \mid X_t)$.
* It decides whether to move to $X^*$ or stay at $X_t$ based on a local acceptance rule $\alpha(X_t, X^*)$.
* After a transient burn-in phase, the sequence of states $\{X_t\}$ mimics draws from $p(x)$: the chain naturally lingers in high-density regions and visits low-density regions rarely.

Unlike rejection sampling, an MCMC step that "rejects" does not throw away computation or restart a loop; it simply sets $X_{t+1} = X_t$, recording the current position again.

---

### Stationarity via detailed balance

Let $T(x \to y)$ be the transition kernel representing the probability (density) of stepping from state $x$ to state $y$.

A distribution $p(x)$ is **stationary** (invariant) under $T$ if running one more step leaves the distribution unchanged:

$$\int p(x) T(x \to y) \, dx = p(y).$$

Directly verifying this global balance integral is difficult. Instead, we enforce a much simpler local condition called **detailed balance** (microscopic reversibility):

$$p(x) T(x \to y) = p(y) T(y \to x) \quad \text{for all } x, y.$$

#### Proof that detailed balance implies stationarity

Integrating both sides with respect to $x$:

$$\int p(x) T(x \to y) \, dx = \int p(y) T(y \to x) \, dx = p(y) \underbrace{\int T(y \to x) \, dx}_{= 1} = p(y).$$

Detailed balance holds, so $p(x)$ is guaranteed to be a stationary distribution.

#### The Metropolis choice

If transitions are generated by proposing $y \sim q(y \mid x)$ and accepting with probability $\alpha(x, y)$, then for $x \neq y$:


$$T(x \to y) = q(y \mid x)\alpha(x, y).$$

To satisfy detailed balance, we set:


$$\alpha(x, y) = \min\left(1, \; \frac{p(y) q(x \mid y)}{p(x) q(y \mid x)}\right) = \min\left(1, \; \frac{f(y) q(x \mid y)}{f(x) q(y \mid x)}\right).$$

The normalizing constant of $p(x)$ cancels completely out of the ratio.

> **Ergodicity guarantee:** Detailed balance guarantees that $p(x)$ is *a* stationary distribution. If the chain is also **irreducible** (any state with $p(x) > 0$ can eventually be reached) and **aperiodic** (it does not cycle deterministically), the ergodic theorem guarantees that the distribution of $X_t$ converges uniquely to $p(x)$ from any starting point $X_0$.

---

### Concrete example: Sampling from a truncated normal

Consider sampling from a standard normal distribution constrained to the interval $[-s, s]$:

$$f(x) = \begin{cases} \exp\left(-\frac{x^2}{2}\right) & \text{if } x \in [-s, s], \\ 0 & \text{otherwise.} \end{cases}$$

#### Random-walk Metropolis formulation

1. **Current state:** $X_t \in [-s, s]$.
2. **Symmetric proposal:** Propose $X^* = X_t + \epsilon$, where $\epsilon \sim \mathcal{N}(0, \sigma_{\text{step}}^2)$. Because $q(X^* \mid X_t) = q(X_t \mid X^*)$, the proposal terms cancel:

$$\alpha(X_t, X^*) = \min\left(1, \; \frac{f(X^*)}{f(X_t)}\right).$$


3. **Acceptance evaluation:**
* If $\vert{}X^*\vert{} > s$, then $f(X^*) = 0 \implies \alpha = 0$ (**instant rejection**; boundary bounce).
* If $\vert{}X^*\vert{} \le s$, then:

$$\alpha = \min\left(1, \; \exp\left(-\frac{(X^*)^2 - X_t^2}{2}\right)\right).$$

* If $\alpha \ge u \sim \operatorname{Uniform}(0, 1)$, set $X_{t+1} = X^*$; otherwise, set $X_{t+1} = X_t$.

#### Autocorrelation function (ACF)

In standard Monte Carlo, every sample is drawn independently: knowing $X_t$ tells you nothing about $X_{t+1}$.

In MCMC, consecutive samples are intrinsically dependent: $X_{t+1}$ depends directly on $X_t$. The **Autocorrelation Function (ACF)** measures how strongly samples separated by a time lag $k$ remain correlated with one another.

For a stationary chain $\{X_1, X_2, \dots, X_N\}$ with sample mean $\bar{X}$ and sample variance $\hat{\sigma}^2$, the sample autocorrelation at **lag $k$** is:

$$\rho_k = \frac{\frac{1}{N - k} \sum_{t=1}^{N - k} (X_t - \bar{X})(X_{t + k} - \bar{X})}{\hat{\sigma}^2}$$

* $\rho_0 = 1.0$ (every sample is perfectly correlated with itself).
* As $k$ grows, $\rho_k$ typically decays toward $0$.
* **Fast decay:** The chain quickly "forgets" its past and explores space efficiently.
* **Slow decay:** The chain is sluggish, heavily autocorrelated, and moves like slow diffusion.

#### Why ACF matters: Effective sample size (ESS)

Because samples are correlated, $N = 10{,}000$ MCMC steps **do not provide $10{,}000$ independent pieces of information**.

The variance of our estimator $\hat{\mu} = \frac{1}{N} \sum_{t=1}^N X_t$ is inflated by the sum of autocorrelations:

$$\operatorname{Var}(\hat{\mu}) \approx \frac{\sigma^2}{N} \underbrace{\left(1 + 2 \sum_{k=1}^\infty \rho_k\right)}_{\text{Autocorrelation Time } \tau_{\text{int}}}$$

The quantity $\tau_{\text{int}} = 1 + 2 \sum_{k=1}^\infty \rho_k$ is the **integrated autocorrelation time**. It defines the **Effective Sample Size (ESS)**:

$$\text{ESS} = \frac{N}{\tau_{\text{int}}} = \frac{N}{1 + 2 \sum_{k=1}^\infty \rho_k}$$

* If $\tau_{\text{int}} = 20$, you need $20$ MCMC steps to get the statistical information equivalent to **$1$ independent draw**.
* If a run has high autocorrelation, an apparently huge sample size ($N = 100{,}000$) might yield an ESS of only a few hundred.

**Task:** Implement the MCMC algorithm above.
1. Run the truncated normal sampler using three distinct values of $\sigma_{\text{step}}$: too small (e.g., $0.02$), optimal ($0.5$), and too large ($5.0$). For each run:
   * Record the overall empirical acceptance rate.
   * Plot the first $1{,}000$ iterations as a trace plot (iteration $t$ vs. value $X_t$).
   * Plot the autocorrelation function (ACF) up to lag $k = 50$.
   * Explain how both extremes inflate autocorrelation and reduce the Effective Sample Size (ESS).
2. Study the lack of stationarity in burn-in samples:
   * Initialize the chain deliberately at the extreme boundary $X_0 = s$. Plot the cumulative running mean $\bar{X}_t = \frac{1}{t} \sum_{i=1}^t X_i$ for $t = 1, \dots, 5{,}000$. Contrast this with the cumulative mean computed after discarding the first $500$ steps.
   * Initialize 4 independent chains with starting points $X_0 \in \{-s, -s/2, s/2, s\}$. Overlay their trace plots for the first $300$ steps. Identify the transition point (iteration number) where the initial conditions are forgotten and the chains fully overlap.

In [4]:
# You're on your own, sorry!

# Problem 3 (3 pts)

Markov Chain Monte Carlo can also be used for sampling from discrete distributions, essentially representing a graph walk.

**Task**: Implement an MCMC scheme for uniformly sampling a random 9x9 Sudoku board. The board should not be completed but must be valid. Pay attention to the balance equation!

In [5]:
# You're on your own, sorry!

# Problem 4 (3 pts)

In Weeks 4 and 5, we studied opinion dynamics on polarized networks with two communities (**Cluster A** and **Cluster B**). When a viral campaign or rumor spreads to every agent in a connected graph $G = (V, E)$ ($|V| = n$, $|E| = m$) without redundant loops, the infection pathways form a **spanning tree** $T \subset E$ ($|T| = n - 1$ edges).

Suppose each edge $e \in E$ has a positive trust/transmission weight $w_e > 0$:
- **Inside each echo chamber ($E_{AA}, E_{BB}$)**: dense, high-trust links ($w_e \in [8, 14]$), including 3 internal **Mega-Hubs** (`[0, 1, 2]`) inside Cluster A.
- **Across the two echo chambers ($E_{AB}$)**:
  - Several "decoy" nodes in Cluster B have many weak, low-trust cross-chamber links ($w_{\text{noise}} = 0.15$ — people who follow an opposing account but ignore it),
  - Only **$k = 3$ hidden Bridge Gatekeepers** in Cluster B have high-trust cross-chamber links ($w_{\text{gate}} = 3.5$) and strong ties into Cluster B's sub-communities.

To move between spanning trees in MCMC, use the **cycle-swap move**:
1. Pick a non-tree edge $e_{\text{in}} \in E \setminus T$ uniformly at random. Adding $e_{\text{in}}$ to $T$ creates a **unique cycle** $C(T, e_{\text{in}})$.
2. Pick an edge $e_{\text{out}} \in C(T, e_{\text{in}})$ uniformly at random from that cycle and remove it, yielding a new spanning tree $T' = T \cup \{e_{\text{in}}\} \setminus \{e_{\text{out}}\}$.

### Your Tasks:

1. **Proposal Symmetry & Sum vs. Product Metropolis Ratios**:
   - Prove that the cycle-swap proposal is symmetric: $q(T' \mid T) = q(T \mid T')$. *(Hint: compare the cycle in $T \cup \{e_{\text{in}}\}$ and the cycle in $T' \cup \{e_{\text{out}}\}$!)*
   - Write down the Metropolis acceptance probability $\alpha(T \to T')$ for two different target distributions over spanning trees:
     1. **Additive Score (Sum of Weights)**: $p_{\text{sum}}(T) \propto \sum_{e \in T} w_e$,
     2. **Multiplicative Score (Product of Weights)**: $p_{\text{prod}}(T) \propto \prod_{e \in T} w_e$.
   - **Explain mathematically** why, for a graph with $n$ vertices, swapping a single edge under $p_{\text{sum}}(T)$ gives a ratio $\frac{f_{\text{sum}}(T')}{f_{\text{sum}}(T)} = 1 + \mathcal{O}(1/n) \approx 1$ that washes out edge weights, whereas under $p_{\text{prod}}(T)$ all $n - 2$ shared edges cancel out!

2. **MCMC Sampling & Echo-Chamber Bridge Bottlenecks**:
   - Implement the cycle-swap MCMC sampler and run it for $N = 25{,}000$ steps (discarding the first $5{,}000$ steps as burn-in) on the network generated below under three targets: **Uniform** ($p_{\text{unif}}(T) \propto 1$), **Sum of Weights** ($p_{\text{sum}}(T)$), and **Product of Weights** ($p_{\text{prod}}(T)$).
   - Plot the empirical distribution of the **number of cross-chamber bridge edges** $b(T) = |T \cap E_{AB}|$ for all three targets, and report their empirical acceptance rates and mean bridge counts $\mathbb{E}[b(T)]$. Why does $p_{\text{prod}}(T)$ collapse the spanning tree onto just $1\text{--}2$ cross-chamber bridges while $p_{\text{sum}}(T)$ behaves like an unweighted uniform tree?

3. **Finding & Buying the $k = 3$ Influencers**:
   - For each MCMC run, rank nodes in Cluster B ($\{40, \dots, 79\}$) by how frequently they serve as endpoints of cross-chamber bridge edges $e \in T \cap E_{AB}$ across the post-burn-in sampled trees.
   - Using `fj_mean_opinion(edges, weights, s_0, lambdas_0, bought_nodes)` provided below, compare the steady-state population opinion $J(S) = \frac{1}{n}\mathbf{1}^\top x(S)$ (and Cluster B's mean opinion) when buying $k = 3$ influencers chosen by:
     1. **Top-3 Weighted Degree** (total incident edge weight $\sum_{v} w_{u,v}$),
     2. **Top-3 Bridge Nodes under $p_{\text{sum}}(T)$**,
     3. **Top-3 Bridge Nodes under $p_{\text{prod}}(T)$**.
   - Which strategy successfully discovers the 3 true Bridge Gatekeepers and flips Cluster B's opinion from negative to positive, and why do the other two fail?

In [ ]:
# You're on your own, sorry!from collections import deque
import numpy as np
import matplotlib.pyplot as plt


def generate_echo_chamber_graph(n_A=40, n_B=40, seed=2026):
    """
    Builds an undirected weighted two-community social network G = (V_A U V_B, E, w)
    with n = n_A + n_B = 80 agents:
      - Cluster A (0 .. n_A-1): already supportive (s_0 in [0.55, 0.85]).
        Nodes 0, 1, 2 are internal Mega-Hubs connected to everyone in Cluster A.
      - Cluster B (n_A .. n-1): opposing echo chamber (s_0 in [-0.80, -0.50]).
        Three hidden Bridge Gatekeepers in Cluster B have high-trust bridge edges
        (w = 3.5) into Cluster A and strong ties into Cluster B's sub-communities,
        while several decoy nodes have many weak cross-chamber noise links (w = 0.15).
    Returns:
      edges:     (m, 2) int array of undirected edges (u < v)
      weights:   (m,) float array of positive edge weights w_e
      s_0:       (n,) baseline convictions in [-1, 1]
      lambdas_0: (n,) baseline susceptibilities in (0, 1)
      T_init:    (n-1,) int array of edge indices forming an initial spanning tree
    """
    rng = np.random.default_rng(seed)
    n = n_A + n_B
    edge_dict = {}

    def add_edge(u, v, w):
        if u == v:
            return
        a, b = (u, v) if u < v else (v, u)
        edge_dict[(a, b)] = max(edge_dict.get((a, b), 0.0), float(w))

    # 1. Cluster A internal backbone + Mega-Hubs (0, 1, 2)
    for i in range(n_A):
        add_edge(i, (i + 1) % n_A, rng.uniform(9.0, 11.0))
        for _ in range(3):
            j = int(rng.integers(0, n_A))
            add_edge(i, j, rng.uniform(8.0, 11.0))
    for hub in [0, 1, 2]:
        for j in range(n_A):
            add_edge(hub, j, rng.uniform(11.0, 14.0))

    # 2. Cluster B internal backbone + 3 sub-communities led by Gatekeepers
    gatekeepers = [n_A, n_A + 1, n_A + 2]
    for i in range(n_A, n):
        add_edge(i, n_A + ((i - n_A + 1) % n_B), rng.uniform(9.0, 11.0))
        for _ in range(2):
            j = int(rng.integers(n_A, n))
            add_edge(i, j, rng.uniform(8.0, 11.0))
    for g_idx, gk in enumerate(gatekeepers):
        for j in range(n_A + 3, n):
            if (j - (n_A + 3)) % 3 == g_idx:
                add_edge(gk, j, rng.uniform(11.0, 13.0))

    # 3. True Gatekeeper bridges across the cut (w = 3.5)
    for g_idx, gk in enumerate(gatekeepers):
        add_edge(3 + 2 * g_idx, gk, 3.5)
        add_edge(4 + 2 * g_idx, gk, 3.5)

    # 4. Decoy "noisy bridge" nodes in Cluster B: many cross-chamber links, but weak weight (w = 0.15)
    noisy_nodes = [n_A + 3, n_A + 4, n_A + 5, n_A + 6, n_A + 7, n_A + 8]
    for nb in noisy_nodes:
        partners = rng.choice(np.arange(10, n_A), size=8, replace=False)
        for u in partners:
            add_edge(int(u), nb, 0.15)

    edges = np.array(sorted(edge_dict.keys()), dtype=int)
    weights = np.array([edge_dict[(u, v)] for u, v in edges], dtype=float)

    s_0 = np.zeros(n)
    s_0[:n_A] = rng.uniform(0.55, 0.85, size=n_A)
    s_0[n_A:] = rng.uniform(-0.80, -0.50, size=n_B)
    lambdas_0 = np.full(n, 0.85)
    lambdas_0[n_A + 3 :] = 0.92

    # Initial spanning tree via BFS
    adj = [[] for _ in range(n)]
    for e_idx, (u, v) in enumerate(edges):
        adj[u].append((v, e_idx))
        adj[v].append((u, e_idx))
    visited = np.zeros(n, dtype=bool)
    visited[0] = True
    q = deque([0])
    T_init = []
    while q:
        u = q.popleft()
        for v, e_idx in adj[u]:
            if not visited[v]:
                visited[v] = True
                T_init.append(e_idx)
                q.append(v)
    T_init = np.array(sorted(T_init), dtype=int)
    return edges, weights, s_0, lambdas_0, T_init


def fj_mean_opinion(edges, weights, s_0, lambdas_0, bought_nodes):
    """
    Evaluates the Friedkin-Johnsen steady-state opinion when `bought_nodes`
    have their convictions pinned to s_c = +1.0 and susceptibility lam_c = 0.01.
    Returns (population_mean_opinion, cluster_B_mean_opinion).
    """
    n = len(s_0)
    A_w = np.zeros((n, n))
    for (u, v), w in zip(edges, weights):
        A_w[u, v] = w
        A_w[v, u] = w
    W = A_w / A_w.sum(axis=1, keepdims=True)
    s = s_0.copy()
    lam = lambdas_0.copy()
    for c in bought_nodes:
        s[c] = 1.0
        lam[c] = 0.01
    x = np.linalg.solve(np.eye(n) - np.diag(lam) @ W, (1.0 - lam) * s)
    return float(x.mean()), float(x[40:].mean())


edges, weights, s_0, lambdas_0, T_init = generate_echo_chamber_graph()

# TODO: Implement cycle-swap spanning-tree MCMC for "unif", "sum", and "prod" targets,
# compare bridge-count distributions, and evaluate the top-3 bought influencers in fj_mean_opinion.

*please don't forget to include your report*